# Hybrid Quantum-Classical Transformer (HQCT) for TSP

This notebook follows the same template as the earlier prototype but implements the proposal architecture:
- Transformer encoder with permutation-invariant city embeddings (no positional encoding)
- Decoder context from graph embedding + first city + last city
- Quantum-enhanced attention head (query-key quantum kernel)
- Enhanced quantum edge scorer (query/key features, deeper PQC)
- Learned blending of classical and quantum logits
- Phase 1 supervised pretraining + Phase 2 REINFORCE fine-tuning with greedy baseline

In [ ]:
# Install dependencies if needed
!pip install torch numpy qiskit qiskit-machine-learning matplotlib tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 6.0 MB/s eta 0:00:00


In [ ]:
import itertools
import math
import random
from dataclasses import dataclass
from typing import List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.neural_networks import EstimatorQNN

import tqdm

## Configuration

`num_cities` controls difficulty. The proposal targets larger TSP (20/50/100), but defaults here are set to run on a single GPU/CPU notebook session.

- Phase 1: supervised pretraining (teacher forcing)
- Phase 2: REINFORCE fine-tuning with greedy rollout baseline

In [ ]:
CONFIG = {
    "num_cities": 7,
    "train_size": 512,
    "val_size": 128,
    "batch_size": 16,
    "hidden_dim": 128,
    "num_heads": 4,
    "num_layers": 2,
    "ffn_dim": 256,
    "dropout": 0.1,
    "num_qubits": 4,
    "epochs_supervised": 8,
    "epochs_reinforce": 2,
    "lr_supervised": 2e-4,
    "lr_reinforce": 1e-4,
    "exact_threshold": 7,
    "seed": 42,
    "use_quantum_attention": True,
    "use_quantum_edge": True,
}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## TSP utilities

In [ ]:
def pairwise_dist(points: np.ndarray) -> np.ndarray:
    diff = points[:, None, :] - points[None, :, :]
    return np.sqrt((diff ** 2).sum(axis=-1))


def tour_length(points: np.ndarray, tour: List[int]) -> float:
    dist = pairwise_dist(points)
    total = 0.0
    for i in range(len(tour)):
        total += dist[tour[i], tour[(i + 1) % len(tour)]]
    return float(total)


def brute_force_tour(points: np.ndarray, start_city: int = 0) -> List[int]:
    n = len(points)
    others = [i for i in range(n) if i != start_city]
    best_len = float("inf")
    best_tour: Optional[List[int]] = None
    for perm in itertools.permutations(others):
        tour = [start_city, *perm]
        cur_len = tour_length(points, tour)
        if cur_len < best_len:
            best_len = cur_len
            best_tour = tour
    assert best_tour is not None
    return best_tour


def nearest_neighbor_tour(points: np.ndarray, start_city: int = 0) -> List[int]:
    n = len(points)
    dist = pairwise_dist(points)
    unvisited = set(range(n))
    unvisited.remove(start_city)
    tour = [start_city]
    cur = start_city
    while unvisited:
        nxt = min(unvisited, key=lambda j: dist[cur, j])
        tour.append(nxt)
        unvisited.remove(nxt)
        cur = nxt
    return tour


def two_opt(points: np.ndarray, tour: List[int]) -> List[int]:
    best = tour[:]
    best_len = tour_length(points, best)
    improved = True
    while improved:
        improved = False
        n = len(best)
        for i in range(1, n - 2):
            for j in range(i + 1, n):
                if j - i == 1:
                    continue
                candidate = best[:]
                candidate[i:j] = reversed(candidate[i:j])
                cand_len = tour_length(points, candidate)
                if cand_len + 1e-12 < best_len:
                    best = candidate
                    best_len = cand_len
                    improved = True
    return best


def build_label_tour(points: np.ndarray, exact_threshold: int = 10) -> Tuple[List[int], bool]:
    n = len(points)
    if n <= exact_threshold:
        return brute_force_tour(points), True
    heuristic = nearest_neighbor_tour(points)
    heuristic = two_opt(points, heuristic)
    return heuristic, False


def describe_difficulty(num_cities: int, exact_threshold: int) -> str:
    if num_cities <= exact_threshold:
        return (
            f"num_cities={num_cities}: exact labels via brute force. "
            f"Good for supervision, expensive as n grows."
        )
    return (
        f"num_cities={num_cities}: labels via nearest-neighbor + 2-opt. "
        f"Scalable but not guaranteed optimal."
    )


def tour_length_torch(points: torch.Tensor, tours: torch.Tensor) -> torch.Tensor:
    ordered = torch.gather(points, 1, tours.unsqueeze(-1).expand(-1, -1, 2))
    shifted = torch.roll(ordered, shifts=-1, dims=1)
    return torch.norm(ordered - shifted, dim=-1).sum(dim=-1)


print(describe_difficulty(CONFIG["num_cities"], CONFIG["exact_threshold"]))

num_cities=7: exact labels via brute force. Good for supervision, expensive as n grows.


## Dataset

In [ ]:
@dataclass
class TSPExample:
    points: np.ndarray
    label_tour: List[int]
    exact: bool


class TSPDataset(Dataset):
    def __init__(self, num_samples: int, num_cities: int, exact_threshold: int = 10, seed: int = 0):
        self.examples: List[TSPExample] = []
        rng = np.random.default_rng(seed)
        for _ in range(num_samples):
            points = rng.random((num_cities, 2), dtype=np.float32)
            label_tour, exact = build_label_tour(points, exact_threshold=exact_threshold)
            self.examples.append(TSPExample(points=points, label_tour=label_tour, exact=exact))

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, idx: int):
        ex = self.examples[idx]
        return {
            "points": torch.tensor(ex.points, dtype=torch.float32),
            "tour": torch.tensor(ex.label_tour, dtype=torch.long),
            "exact": torch.tensor(int(ex.exact), dtype=torch.long),
        }


def collate_fn(batch):
    points = torch.stack([b["points"] for b in batch], dim=0)
    tours = torch.stack([b["tour"] for b in batch], dim=0)
    exact = torch.stack([b["exact"] for b in batch], dim=0)
    return points, tours, exact

## Quantum components (proposal-aligned)

- Quantum attention head: query-key quantum kernel
- Enhanced quantum edge scorer: query/key features, deeper PQC with extra `Rz` and second CZ layer

In [ ]:
class QuantumKernelCircuit(nn.Module):
    """Shared PQC block for quantum attention/edge scoring."""

    def __init__(self, input_dim: int, num_qubits: int = 4):
        super().__init__()
        self.num_qubits = num_qubits
        self.in_proj = nn.Linear(input_dim, num_qubits)

        x = ParameterVector("x", num_qubits)
        theta = ParameterVector("theta", num_qubits)
        phi = ParameterVector("phi", num_qubits)

        qc = QuantumCircuit(num_qubits)
        for i in range(num_qubits):
            qc.rx(x[i], i)
            qc.ry(theta[i], i)
            qc.rz(phi[i], i)

        for i in range(num_qubits):
            qc.cz(i, (i + 1) % num_qubits)

        if num_qubits > 2:
            for i in range(num_qubits):
                qc.cz(i, (i + 2) % num_qubits)

        observable = SparsePauliOp.from_list([("Z" * num_qubits, 1.0)])

        qnn = EstimatorQNN(
            circuit=qc,
            input_params=list(x),
            weight_params=list(theta) + list(phi),
            observables=observable,
            input_gradients=True,
        )
        self.q_layer = TorchConnector(qnn)
        self.out_proj = nn.Linear(1, 1)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        q_inputs = math.pi * torch.tanh(self.in_proj(features))
        q_out = self.q_layer(q_inputs)
        if q_out.dim() == 1:
            q_out = q_out.unsqueeze(-1)
        return self.out_proj(q_out).squeeze(-1)


class QuantumAttentionHead(nn.Module):
    """Quantum kernel score for decoder query vs encoder keys."""

    def __init__(self, hidden_dim: int, num_qubits: int = 4):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.kernel = QuantumKernelCircuit(input_dim=2 * hidden_dim, num_qubits=num_qubits)

    def forward(self, query: torch.Tensor, keys: torch.Tensor) -> torch.Tensor:
        bsz, num_nodes, hidden_dim = keys.shape
        q = query.unsqueeze(1).expand(-1, num_nodes, -1)
        feats = torch.cat([q, keys], dim=-1).reshape(bsz * num_nodes, 2 * hidden_dim)
        return self.kernel(feats).reshape(bsz, num_nodes)


class EnhancedQuantumEdgeScorer(nn.Module):
    """Proposal-aligned edge scorer using query/key projections (not raw coordinates)."""

    def __init__(self, hidden_dim: int, num_qubits: int = 4):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.kernel = QuantumKernelCircuit(input_dim=2 * hidden_dim, num_qubits=num_qubits)

    def forward(self, query: torch.Tensor, keys: torch.Tensor) -> torch.Tensor:
        bsz, num_nodes, hidden_dim = keys.shape
        q = query.unsqueeze(1).expand(-1, num_nodes, -1)
        feats = torch.cat([q, keys], dim=-1).reshape(bsz * num_nodes, 2 * hidden_dim)
        return self.kernel(feats).reshape(bsz, num_nodes)

## HQCT model

The encoder is a Transformer stack, and the decoder uses proposal-style context (`graph`, `first city`, `last city`) to score feasible next cities with blended classical and quantum logits.

In [ ]:
class HQCTPointer(nn.Module):
    def __init__(
        self,
        hidden_dim: int = 128,
        num_heads: int = 4,
        num_layers: int = 2,
        ffn_dim: int = 256,
        dropout: float = 0.1,
        num_qubits: int = 4,
        use_quantum_attention: bool = True,
        use_quantum_edge: bool = True,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.use_quantum_attention = use_quantum_attention
        self.use_quantum_edge = use_quantum_edge

        self.city_embed = nn.Linear(2, hidden_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.context_proj = nn.Sequential(
            nn.Linear(3 * hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.query_proj = nn.Linear(hidden_dim, hidden_dim)
        self.key_proj = nn.Linear(hidden_dim, hidden_dim)

        self.classical_scale = nn.Parameter(torch.tensor(1.0))
        self.alpha = nn.Parameter(torch.tensor(1.0))
        self.beta = nn.Parameter(torch.tensor(1.0))

        self.q_attention_head = QuantumAttentionHead(hidden_dim=hidden_dim, num_qubits=num_qubits)
        self.q_edge_scorer = EnhancedQuantumEdgeScorer(hidden_dim=hidden_dim, num_qubits=num_qubits)

    def _encode(self, points: torch.Tensor) -> torch.Tensor:
        emb = self.city_embed(points)
        return self.encoder(emb)

    def _build_query(
        self,
        enc_states: torch.Tensor,
        first_city_idx: torch.Tensor,
        last_city_idx: torch.Tensor,
    ) -> torch.Tensor:
        bsz = enc_states.size(0)
        batch_idx = torch.arange(bsz, device=enc_states.device)
        graph_emb = enc_states.mean(dim=1)
        first_emb = enc_states[batch_idx, first_city_idx]
        last_emb = enc_states[batch_idx, last_city_idx]
        context = torch.cat([graph_emb, first_emb, last_emb], dim=-1)
        return self.query_proj(self.context_proj(context))

    def _step_logits(
        self,
        query: torch.Tensor,
        enc_states: torch.Tensor,
        visited_mask: torch.Tensor,
    ) -> torch.Tensor:
        keys = self.key_proj(enc_states)
        classical_logits = self.classical_scale * torch.einsum("bd,bnd->bn", query, keys) / math.sqrt(self.hidden_dim)

        quantum_logits = torch.zeros_like(classical_logits)
        if self.use_quantum_attention:
            quantum_logits = quantum_logits + self.q_attention_head(query, keys)
        if self.use_quantum_edge:
            quantum_logits = quantum_logits + self.q_edge_scorer(query, keys)

        alpha = F.softplus(self.alpha)
        beta = F.softplus(self.beta)
        logits = alpha * classical_logits + beta * quantum_logits
        logits = logits.masked_fill(visited_mask, -1e9)
        return logits

    def _decode(
        self,
        points: torch.Tensor,
        target_tour: Optional[torch.Tensor] = None,
        decode_type: str = "greedy",
    ):
        bsz, num_nodes, _ = points.shape
        enc_states = self._encode(points)

        visited = torch.zeros(bsz, num_nodes, dtype=torch.bool, device=points.device)
        first_city = torch.zeros(bsz, dtype=torch.long, device=points.device)
        current_city = first_city.clone()
        visited[:, 0] = True

        pred_tours = [current_city]
        logits_steps = []
        logprob_sum = torch.zeros(bsz, device=points.device)

        for step in range(num_nodes - 1):
            query = self._build_query(enc_states, first_city, current_city)
            logits = self._step_logits(query, enc_states, visited)
            logits_steps.append(logits)

            if target_tour is not None:
                next_city = target_tour[:, step + 1]
            elif decode_type == "greedy":
                next_city = logits.argmax(dim=-1)
            elif decode_type == "sampling":
                dist = torch.distributions.Categorical(logits=logits)
                next_city = dist.sample()
                logprob_sum = logprob_sum + dist.log_prob(next_city)
            else:
                raise ValueError(f"Unknown decode_type: {decode_type}")

            pred_tours.append(next_city)
            visited = visited.clone()
            visited[torch.arange(bsz, device=points.device), next_city] = True
            current_city = next_city

        logits_steps = torch.stack(logits_steps, dim=1)
        pred_tours = torch.stack(pred_tours, dim=1)
        return logits_steps, pred_tours, logprob_sum

    def forward(self, points: torch.Tensor, target_tour: torch.Tensor):
        return self._decode(points, target_tour=target_tour, decode_type="greedy")[:2]

    @torch.no_grad()
    def greedy_decode(self, points: torch.Tensor) -> torch.Tensor:
        _, tours, _ = self._decode(points, target_tour=None, decode_type="greedy")
        return tours

    def sample_decode(self, points: torch.Tensor):
        _, tours, logprob_sum = self._decode(points, target_tour=None, decode_type="sampling")
        return tours, logprob_sum

## Training and evaluation

- **Phase 1 (Supervised):** teacher forcing + cross-entropy
- **Phase 2 (REINFORCE):** sampled tours, greedy rollout baseline

In [ ]:
def train_supervised_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    total_count = 0

    for points, tours, _ in tqdm.tqdm(loader, desc="Supervised"):
        points = points.to(device)
        tours = tours.to(device)

        logits_steps, _ = model(points, target_tour=tours)
        targets = tours[:, 1:]
        loss = F.cross_entropy(logits_steps.reshape(-1, logits_steps.size(-1)), targets.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item()) * points.size(0)
        total_count += points.size(0)

    return total_loss / max(total_count, 1)


def train_reinforce_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    total_adv = 0.0
    total_count = 0

    for points, _, _ in tqdm.tqdm(loader, desc="REINFORCE"):
        points = points.to(device)

        sampled_tours, logprob_sum = model.sample_decode(points)
        with torch.no_grad():
            greedy_tours = model.greedy_decode(points)

        sampled_len = tour_length_torch(points, sampled_tours)
        baseline_len = tour_length_torch(points, greedy_tours)
        advantage = sampled_len - baseline_len

        loss = (advantage.detach() * logprob_sum).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item()) * points.size(0)
        total_adv += float(advantage.mean().item()) * points.size(0)
        total_count += points.size(0)

    return {
        "reinforce_loss": total_loss / max(total_count, 1),
        "avg_advantage": total_adv / max(total_count, 1),
    }


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()

    total_pred_len = 0.0
    total_label_len = 0.0
    total_exact_gap = 0.0
    exact_count = 0
    count = 0

    for points, tours, exact in loader:
        points = points.to(device)
        tours = tours.to(device)

        pred_tours = model.greedy_decode(points)
        pred_len = tour_length_torch(points, pred_tours)
        label_len = tour_length_torch(points, tours)

        total_pred_len += float(pred_len.sum().item())
        total_label_len += float(label_len.sum().item())

        exact_mask = exact.to(device).bool()
        if exact_mask.any():
            gap = pred_len[exact_mask] - label_len[exact_mask]
            total_exact_gap += float(gap.sum().item())
            exact_count += int(exact_mask.sum().item())

        count += points.size(0)

    avg_pred = total_pred_len / max(count, 1)
    avg_label = total_label_len / max(count, 1)
    avg_exact_gap = total_exact_gap / max(exact_count, 1) if exact_count > 0 else None
    gap_pct = 100.0 * (avg_pred - avg_label) / max(avg_label, 1e-9)

    return {
        "avg_pred_tour_length": avg_pred,
        "avg_label_tour_length": avg_label,
        "optimality_gap_pct_vs_labels": gap_pct,
        "avg_exact_gap": avg_exact_gap,
        "exact_eval_count": exact_count,
    }

## Build datasets and model

In [ ]:
train_ds = TSPDataset(
    num_samples=CONFIG["train_size"],
    num_cities=CONFIG["num_cities"],
    exact_threshold=CONFIG["exact_threshold"],
    seed=CONFIG["seed"],
)

val_ds = TSPDataset(
    num_samples=CONFIG["val_size"],
    num_cities=CONFIG["num_cities"],
    exact_threshold=CONFIG["exact_threshold"],
    seed=CONFIG["seed"] + 1,
)

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, collate_fn=collate_fn)

model = HQCTPointer(
    hidden_dim=CONFIG["hidden_dim"],
    num_heads=CONFIG["num_heads"],
    num_layers=CONFIG["num_layers"],
    ffn_dim=CONFIG["ffn_dim"],
    dropout=CONFIG["dropout"],
    num_qubits=CONFIG["num_qubits"],
    use_quantum_attention=CONFIG["use_quantum_attention"],
    use_quantum_edge=CONFIG["use_quantum_edge"],
).to(device)

optimizer_sup = torch.optim.Adam(model.parameters(), lr=CONFIG["lr_supervised"])
optimizer_rl = torch.optim.Adam(model.parameters(), lr=CONFIG["lr_reinforce"])

print(describe_difficulty(CONFIG["num_cities"], CONFIG["exact_threshold"]))
print(model.__class__.__name__)

/tmp/ipykernel_20520/2663327114.py:29: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


num_cities=7: exact labels via brute force. Good for supervision, expensive as n grows.
HQCTPointer


## Train (Phase 1 and Phase 2)

In [ ]:
for epoch in range(1, CONFIG["epochs_supervised"] + 1):
    train_loss = train_supervised_epoch(model, train_loader, optimizer_sup, device)
    metrics = evaluate(model, val_loader, device)
    exact_gap = metrics["avg_exact_gap"]
    exact_gap_str = f"{exact_gap:.4f}" if exact_gap is not None else "N/A"

    print(
        f"[Supervised] Epoch {epoch:02d} | "
        f"loss={train_loss:.4f} | "
        f"pred_len={metrics['avg_pred_tour_length']:.4f} | "
        f"label_len={metrics['avg_label_tour_length']:.4f} | "
        f"gap%={metrics['optimality_gap_pct_vs_labels']:.2f} | "
        f"exact_gap={exact_gap_str}"
    )

for epoch in range(1, CONFIG["epochs_reinforce"] + 1):
    rl_stats = train_reinforce_epoch(model, train_loader, optimizer_rl, device)
    metrics = evaluate(model, val_loader, device)
    exact_gap = metrics["avg_exact_gap"]
    exact_gap_str = f"{exact_gap:.4f}" if exact_gap is not None else "N/A"

    print(
        f"[REINFORCE] Epoch {epoch:02d} | "
        f"policy_loss={rl_stats['reinforce_loss']:.4f} | "
        f"adv={rl_stats['avg_advantage']:.4f} | "
        f"pred_len={metrics['avg_pred_tour_length']:.4f} | "
        f"label_len={metrics['avg_label_tour_length']:.4f} | "
        f"gap%={metrics['optimality_gap_pct_vs_labels']:.2f} | "
        f"exact_gap={exact_gap_str}"
    )

Supervised: 100%|██████████| 32/32 [08:20<00:00, 15.64s/it]


[Supervised] Epoch 01 | loss=1.0852 | pred_len=2.8234 | label_len=2.4927 | gap%=13.27 | exact_gap=0.3307


Supervised: 100%|██████████| 32/32 [08:19<00:00, 15.61s/it]


[Supervised] Epoch 02 | loss=0.8153 | pred_len=2.5842 | label_len=2.4927 | gap%=3.67 | exact_gap=0.0915


Supervised: 100%|██████████| 32/32 [08:19<00:00, 15.60s/it]


...

[Supervised] Epoch 08 | loss=0.4185 | pred_len=2.5275 | label_len=2.4927 | gap%=1.40 | exact_gap=0.0348


REINFORCE: 100%|██████████| 32/32 [08:36<00:00, 16.15s/it]


[REINFORCE] Epoch 01 | policy_loss=-0.2528 | adv=0.0600 | pred_len=2.5465 | label_len=2.4927 | gap%=2.16 | exact_gap=0.0538


REINFORCE: 100%|██████████| 32/32 [08:36<00:00, 16.15s/it]


[REINFORCE] Epoch 02 | policy_loss=-0.1090 | adv=0.0294 | pred_len=2.5426 | label_len=2.4927 | gap%=2.00 | exact_gap=0.0499


## Inspect a few predictions

In [ ]:
points, tours, exact = next(iter(val_loader))
points_device = points.to(device)
pred_tours = model.greedy_decode(points_device).cpu()

for i in range(min(3, len(pred_tours))):
    pts = points[i].numpy()
    pred = pred_tours[i].tolist()
    label = tours[i].tolist()

    print(f"Instance {i} | exact_label={bool(exact[i].item())}")
    print("  pred_tour :", pred, "len=", round(tour_length(pts, pred), 4))
    print("  label_tour:", label, "len=", round(tour_length(pts, label), 4))
    print()

Instance 0 | exact_label=True
  pred_tour : [0, 3, 4, 1, 2, 5, 6] len= 2.6681
  label_tour: [0, 4, 1, 2, 5, 6, 3] len= 2.6518

Instance 1 | exact_label=True
  pred_tour : [0, 5, 1, 3, 4, 2, 6] len= 2.2804
  label_tour: [0, 5, 1, 3, 4, 2, 6] len= 2.2804

Instance 2 | exact_label=True
  pred_tour : [0, 3, 4, 5, 2, 1, 6] len= 3.1149
  label_tour: [0, 1, 2, 6, 5, 4, 3] len= 2.8507



In [ ]:
# # Optional Phase 3 scaffold: run quantum components on IBM hardware
# ENABLE_IBM_HARDWARE = False

# if ENABLE_IBM_HARDWARE:
#     # Requires qiskit-ibm-runtime and authenticated IBM Quantum account.
#     from qiskit_ibm_runtime import QiskitRuntimeService

#     service = QiskitRuntimeService()
#     backends = service.backends(simulator=False, operational=True)
#     print("Available hardware backends:", [b.name for b in backends][:10])
# else:
#     print("IBM hardware run disabled. Set ENABLE_IBM_HARDWARE=True after configuring credentials.")

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, Session, Estimator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

In [ ]:
service = QiskitRuntimeService(instance="crn:v1:bluemix:public:quantum-computing:us-east:a/b8ff6077c08a4ea9871560ccb827d457:d3452110-b228-4c79-8959-15ea8cfd435d::")
backend = service.backend("ibm_rensselaer")

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, Session, Estimator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.neural_networks import EstimatorQNN

# 1. Connect
service = QiskitRuntimeService(instance="crn:v1:bluemix:public:quantum-computing:us-east:a/b8ff6077c08a4ea9871560ccb827d457:d3452110-b228-4c79-8959-15ea8cfd435d::")
backend = service.backend("ibm_rensselaer")
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

def rebuild_qnn(kernel_module, estimator):
    """Transpile and rebuild a QuantumKernelCircuit's QNN in-place."""
    qnn = kernel_module.q_layer._neural_network
    isa_circuit = pm.run(qnn.circuit)
    isa_observable = qnn.observables[0].apply_layout(isa_circuit.layout)

    new_qnn = EstimatorQNN(
        circuit=isa_circuit,
        input_params=qnn.input_params,
        weight_params=qnn.weight_params,
        observables=isa_observable,
        input_gradients=True,
        estimator=estimator,
    )
    kernel_module.q_layer = TorchConnector(new_qnn)

# 2. Open session and rebuild both quantum components
with Session(backend=backend, max_time=28800) as session:
    estimator = Estimator(mode=session)

    rebuild_qnn(model.q_attention_head.kernel, estimator)
    rebuild_qnn(model.q_edge_scorer.kernel, estimator)

    # 3. Evaluate
    metrics = evaluate(model, val_loader, device)
    print("Hardware eval:", metrics)

## Notes

- This notebook is proposal-aligned at architecture/training-flow level (HQCT with quantum head + enhanced edge scorer).
- For large-scale experiments (TSP20/50/100), replace synthetic labels with Concorde/LKH-3 pipelines and increase dataset sizes.
- The quantum parts can be expensive; for quick debugging, reduce `num_cities`, batch size, or disable one quantum component.